# Chapter 20 — LLM Sandbox — 위험 도구·Agent 격리 실행

> **Chapter 20 · Part III**

`subprocess`로 환경변수와 자원 사용을 제한해 봅니다. 빈 환경변수·임시 작업디렉터리·`resource.setrlimit`(CPU·메모리)·timeout 을 걸어, 환경변수 노출·자원 폭주의 차이를 실측하고, 별도의 경로 검사 함수를 확인합니다. 파일시스템·네트워크 접근 자체는 격리하지 않습니다.

## 실습 목표
1. 격리 없이 실행하면 부모의 비밀 환경변수가 자식 프로세스로 새는 것을 확인하고, 빈 env 로 차단합니다.
2. 무한 루프를 CPU 상한(`RLIMIT_CPU`)·timeout 으로 강제 종료하고, 메모리 상한(`RLIMIT_AS`)으로 폭주 할당을 막습니다.
3. `os.path.realpath` 기반 검사 함수가 허용 경로 밖의 요청을 거부하는지 확인합니다. 자식 프로세스의 임의 파일 접근을 차단하는 기능은 아닙니다.

> **주의:** 이 실험은 Colab/Linux 의 프로세스 수준 격리입니다(`resource`·`preexec_fn` 은 POSIX 전용). 프로덕션급 커널 격리는 gVisor·Firecracker 같은 인프라가 담당하며(Step 6 개념 설명), 여기서 실행하는 위험 코드는 모두 무해한 데모입니다.

In [1]:
# 표준 라이브러리만으로 동작 — 설치 불필요
# (resource·preexec_fn 은 Linux/Colab 전용. gVisor·Firecracker 는 인프라 — 노트북 밖)
import os, sys, signal, resource, subprocess, tempfile, time

In [2]:
# 실행 제한과 미구현 범위 — CPU·메모리·timeout만 강제한다
SANDBOX_POLICY = {
    'network': 'not-isolated',  # 이 헬퍼는 네트워크를 차단하지 않음
    'filesystem': 'cwd-only',   # 작업디렉터리만 변경; 다른 경로 접근 가능
    'cpu_seconds': 1,           # RLIMIT_CPU 소프트 상한
    'mem_bytes': 512 * 1024 * 1024,  # RLIMIT_AS 512MB
    'timeout': 5,               # 벽시계 timeout(초)
}
SANDBOX_POLICY

{'network': 'not-isolated',
 'filesystem': 'cwd-only',
 'cpu_seconds': 1,
 'mem_bytes': 536870912,
 'timeout': 5}

---
## Step 1. 프로세스 격리 헬퍼

위험 도구를 **부모와 분리된 자식 프로세스**로 실행하는 헬퍼를 만듭니다. `sandboxed=True` 면
빈 환경변수·임시 작업디렉터리·CPU/메모리 `setrlimit`·timeout 을 모두 걸고, `False` 면 부모
환경을 그대로 물려받아 실행합니다. 이후 스텝에서 같은 위험 코드를 두 모드로 돌려 차이를
실측합니다.

In [3]:
def _apply_limits(policy):
    """자식 프로세스가 exec 되기 직전(preexec_fn)에 커널 자원 상한을 건다."""
    # CPU: 소프트 1초에서 SIGXCPU, 하드 2초에서 SIGKILL
    resource.setrlimit(resource.RLIMIT_CPU, (policy['cpu_seconds'], policy['cpu_seconds'] + 1))
    # 가상 메모리(주소공간) 상한
    resource.setrlimit(resource.RLIMIT_AS, (policy['mem_bytes'], policy['mem_bytes']))

def run_child(code: str, sandboxed: bool, policy=SANDBOX_POLICY):
    """위험 코드를 별도 파이썬 프로세스로 실행. sandboxed 면 env·cwd·rlimit·timeout 격리."""
    env = {} if sandboxed else dict(os.environ)             # 빈 env vs 부모 env 상속
    cwd = tempfile.mkdtemp() if sandboxed else os.getcwd()  # 임시 디렉터리 vs 현재 디렉터리
    preexec = (lambda: _apply_limits(policy)) if sandboxed else None
    t0 = time.time()
    try:
        p = subprocess.run([sys.executable, '-c', code], env=env, cwd=cwd,
                           preexec_fn=preexec, capture_output=True, text=True,
                           timeout=policy['timeout'])
        killed = signal.Signals(-p.returncode).name if p.returncode < 0 else ''  # SIGXCPU/SIGKILL
        err_lines = p.stderr.strip().splitlines()
        return {'rc': p.returncode, 'killed_by': killed, 'timeout': False,
                'out': p.stdout.strip(), 'err': err_lines[-1] if err_lines else '',
                'wall': round(time.time() - t0, 2)}
    except subprocess.TimeoutExpired:
        return {'rc': None, 'killed_by': 'TIMEOUT', 'timeout': True,
                'out': '', 'err': f"timeout {policy['timeout']}s 초과 — 강제 종료",
                'wall': round(time.time() - t0, 2)}
print('run_child 헬퍼 준비 완료')

run_child 헬퍼 준비 완료


---
## Step 2. 비밀 환경변수 노출 대비

부모 프로세스에 API 키·DB 비밀번호 같은 값이 환경변수로 들어 있으면, 위험 도구를 그냥
실행할 때 자식이 그 값을 그대로 읽습니다. 빈 `env={}` 로 격리하면 새지 않습니다.

In [4]:
# 부모 프로세스에 민감정보가 환경변수로 있다고 가정
os.environ['SANDBOX_DEMO_SECRET'] = 'db-password-1234'

# 위험 도구가 몰래 환경변수를 훔치려 한다
steal = "import os; print('훔친 값:', os.environ.get('SANDBOX_DEMO_SECRET'))"

no_box = run_child(steal, sandboxed=False)
box    = run_child(steal, sandboxed=True)
print('[격리 없음] 자식 출력:', no_box['out'], '  <- 비밀이 그대로 노출')
print('[격리 적용] 자식 출력:', box['out'], '     <- 빈 env 라 None')
assert 'db-password-1234' in no_box['out'] and 'None' in box['out']

[격리 없음] 자식 출력: 훔친 값: db-password-1234   <- 비밀이 그대로 노출
[격리 적용] 자식 출력: 훔친 값: None      <- 빈 env 라 None


---
## Step 3. 무한 루프 — timeout + CPU 상한

프롬프트 인젝션으로 "끝나지 않는 계산"을 시키면, 격리가 없으면 프로세스가 영원히 CPU 를
잡습니다. 격리 모드에서는 `RLIMIT_CPU` 가 커널 수준에서 프로세스를 죽이고, timeout 이
마지막 안전망이 됩니다. 격리 없이 돌리면 timeout 이 도달할 때까지 멈추지 않는 것으로
"스스로 끝나지 않음"을 확인합니다.

In [5]:
infinite = "while True:\n    pass"

no_box = run_child(infinite, sandboxed=False)   # CPU 상한 없음 → timeout 까지 지속
box    = run_child(infinite, sandboxed=True)    # RLIMIT_CPU 가 먼저 죽임

print(f"[격리 없음] timeout={no_box['timeout']}  {no_box['err']}  (약 {no_box['wall']}s 뒤 강제 종료)")
print(f"[격리 적용] killed_by={box['killed_by']}  rc={box['rc']}  wall={box['wall']}s  <- 커널이 CPU 상한으로 종료")
assert no_box['timeout'] is True          # 스스로 끝나지 않아 timeout 으로만 멈춤
assert box['killed_by'] in ('SIGXCPU', 'SIGKILL') and box['wall'] < no_box['wall']

[격리 없음] timeout=True  timeout 5s 초과 — 강제 종료  (약 5.0s 뒤 강제 종료)
[격리 적용] killed_by=SIGXCPU  rc=-24  wall=1.0s  <- 커널이 CPU 상한으로 종료


---
## Step 4. 메모리 상한 — 폭주 할당 차단

메모리 폭탄(대용량 할당)은 격리가 없으면 호스트 RAM 을 통째로 삼킵니다. 여기서는 공유
서버에 부담을 주지 않도록 **격리 모드만** 실행합니다 — `RLIMIT_AS`(512MB) 가 걸려 있어
1GB 할당 시도는 즉시 `MemoryError` 로 막히고, 실제로 큰 메모리를 잡지 않습니다.

In [6]:
# 1GB 할당 시도 — RLIMIT_AS(512MB) 하한에 걸려 즉시 실패해야 한다
mem_bomb = "b = bytearray(1024 * 1024 * 1024); print('할당 성공', len(b))"

box = run_child(mem_bomb, sandboxed=True)
print(f"[격리 적용] rc={box['rc']}  err={box['err']}  <- RLIMIT_AS 로 즉시 차단(실제 1GB 잡지 않음)")
assert box['rc'] != 0 and 'MemoryError' in box['err']

[격리 적용] rc=1  err=MemoryError  <- RLIMIT_AS 로 즉시 차단(실제 1GB 잡지 않음)


---
## Step 5. 경로 탈출(`../`) 차단 — realpath

파일 접근을 특정 작업 디렉터리로 제한해도, `../../etc/passwd` 같은 상대 경로로 밖으로
빠져나가려는 시도가 흔합니다. `os.path.realpath` 로 심볼릭 링크와 `..` 를 모두 풀어 정규화한
뒤, 그 결과가 허용 디렉터리 안에 있는지 확인하면 탈출을 잡습니다.

In [7]:
SANDBOX_ROOT = os.path.realpath(tempfile.mkdtemp(prefix='agent_workdir_'))

def resolve_in_sandbox(user_path: str) -> str:
    """user_path 를 정규화해 SANDBOX_ROOT 밖이면 거부한다."""
    full = os.path.realpath(os.path.join(SANDBOX_ROOT, user_path))
    if full != SANDBOX_ROOT and not full.startswith(SANDBOX_ROOT + os.sep):
        raise PermissionError(f'경로 탈출 차단: {user_path}')
    return full

print('작업 디렉터리(SANDBOX_ROOT):', SANDBOX_ROOT)
for p in ['report.txt', 'sub/data.json', '../../../etc/passwd', '/etc/shadow']:
    try:
        resolved = resolve_in_sandbox(p)
        print(f"  허용  {p:24} → {resolved}")
    except PermissionError as e:
        print(f"  차단  {p:24} → {e}")

작업 디렉터리(SANDBOX_ROOT): /tmp/agent_workdir_h2mr6cjr
  허용  report.txt               → /tmp/agent_workdir_h2mr6cjr/report.txt
  허용  sub/data.json            → /tmp/agent_workdir_h2mr6cjr/sub/data.json
  차단  ../../../etc/passwd      → 경로 탈출 차단: ../../../etc/passwd
  차단  /etc/shadow              → 경로 탈출 차단: /etc/shadow


---
## Step 6. 커널 격리는 인프라 — gVisor·Firecracker (개념)

위 프로세스 격리(빈 env·rlimit·realpath)는 **같은 커널을 공유**합니다. 커널 자체를 노리는
공격(커널 취약점을 통한 sandbox escape)은 프로세스 상한으로 막을 수 없습니다. 그래서
프로덕션에서는 커널을 분리하는 인프라를 덧댑니다 — 본문 §20.3 참고.

- **gVisor**: syscall 을 사용자공간 커널(Sentry)이 가로채, 자식이 호스트 커널을 직접
  건드리지 못하게 합니다(Docker 호환 런타임).
- **Firecracker / Kata**: 도구 호출마다 초경량 microVM 을 띄워 커널 자체를 분리하고, 짧은
  TTL 로 폐기합니다.

이 노트북의 상한·경로 검사는 그 인프라 안에서도 **defense-in-depth 의 안쪽 겹**으로 그대로
유효합니다.

---
## 정리
- 위험 도구·Agent 는 격리 프로세스에서 실행하고, **빈 env·임시 작업디렉터리·`setrlimit`(CPU·메모리)·timeout** 으로 환경변수 상속과 자원 사용을 제한합니다. 파일에 저장된 비밀이나 네트워크 접근은 차단하지 않습니다.
- 경로 검사 함수는 그 함수를 거치는 요청만 검사합니다. 자식 프로세스의 직접 파일 접근을 막으려면 별도의 OS 격리가 필요합니다.
- 이 프로세스 격리는 같은 커널을 공유하므로, 커널 자체를 노리는 escape 는 gVisor·Firecracker 같은 인프라(Step 6)로 막습니다. 두 겹을 함께 두는 것이 defense-in-depth 입니다.

> **다음 챕터:** Federated Threat Intelligence.